# Fine-tuning with QLoRA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DSPagan/llms-time-complexity/blob/main/notebooks/fine_tuning.ipynb)

Fine-tune `Llama 3.1 8B Instruct` (4-bit) on the CodeComplex training split with QLoRA, then evaluate on the test set. The logic lives in `src/`; this notebook just wires it together.

In [ ]:
# Unsloth pulls its own compatible stack; Colab already provides a CUDA-enabled PyTorch.
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo

In [ ]:
# Clone the repo (code + data) and regenerate the train/test split from CodeComplex.
!git clone https://github.com/DSPagan/llms-time-complexity.git
%cd llms-time-complexity
!python src/prepare_data.py

In [ ]:
import os, sys, json

sys.path.insert(0, os.getcwd())

from unsloth import FastLanguageModel
from src.load_model import load_model
from src.train_model import train_model
from src.prompts import build_prompt
from src.evaluate import evaluate, plot_confusion_matrix, print_summary

In [ ]:
max_seq_length = 2048
model, tokenizer = load_model(max_seq_length=max_seq_length)

## Fine-tuning with QLoRA

Two epochs over the training split (the best configuration in the thesis). `train_model` returns the fine-tuned model, so we capture it for evaluation.

In [ ]:
model, trainer_stats = train_model(
    "data/train_data.jsonl", model, tokenizer,
    num_epochs=2, max_seq_length=max_seq_length,
)

## Evaluation

Run the fine-tuned model over the test set and compute accuracy, macro F1 and the confusion matrix. The fine-tuned model answers with the class name, which `evaluate` maps to the canonical classes.

In [ ]:
FastLanguageModel.for_inference(model)

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

test_data = read_jsonl("data/test_data.jsonl")
y_true = [item["complexity"] for item in test_data]

def predict(src, max_new_tokens=64):
    messages = [{"role": "user", "content": build_prompt(src)}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    if inputs.shape[1] > max_seq_length:
        return None
    out = model.generate(
        input_ids=inputs, do_sample=False, max_new_tokens=max_new_tokens,
        use_cache=True, no_repeat_ngram_size=4,
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split("assistant")[-1].strip()

raw = [predict(item["src"]) for item in test_data]
res = evaluate(y_true, raw)
print_summary("fine-tuned (2 epochs)", res)

os.makedirs("figures", exist_ok=True)
plot_confusion_matrix(res["confusion_matrix"], title="Fine-tuned (QLoRA, 2 epochs)",
                      save_path="figures/CM_finetuned.png")